# Cleaning and Understanding the Dataset - Bank Customer Churn

In [16]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler


df = pd.read_csv('../data/Bank Customer Churn Prediction.csv')
df.head()

,customer_id,credit_score,country,gender,age,tenure,balance,products_number,credit_card,active_member,estimated_salary,churn
0,15634602,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,15647311,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,15619304,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,15701354,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,15737888,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


## Dataset shape and Churn rate

### First check to see if there is any missing or anomolous data:

In [17]:
print(f"Dataset: {df.shape[0]:,} customers, {df.shape[1]} features")
print(f"Missing values: {df.isnull().sum().sum():,}")
print(f"\nChecking for anomalies:\n {df.describe()}")
# No missing data and there aren't any anomalies in the dataset from what we can see below.


Dataset: 10,000 customers, 12 features
Missing values: 0

Checking for anomalies:
         customer_id  credit_score           age        tenure        balance  \
count  1.000000e+04  10000.000000  10000.000000  10000.000000   10000.000000   
mean   1.569094e+07    650.528800     38.921800      5.012800   76485.889288   
std    7.193619e+04     96.653299     10.487806      2.892174   62397.405202   
min    1.556570e+07    350.000000     18.000000      0.000000       0.000000   
25%    1.562853e+07    584.000000     32.000000      3.000000       0.000000   
50%    1.569074e+07    652.000000     37.000000      5.000000   97198.540000   
75%    1.575323e+07    718.000000     44.000000      7.000000  127644.240000   
max    1.581569e+07    850.000000     92.000000     10.000000  250898.090000   

       products_number  credit_card  active_member  estimated_salary  \
count     10000.000000  10000.00000   10000.000000      10000.000000   
mean          1.530200      0.70550       0.515100  

### Exploring and Encoding Categorical Features

In [18]:
#Separating categorical columns
categorical_cols = df.select_dtypes(include=['object', 'category']).columns
print(f"Categorical features found: {list(categorical_cols)}")

#Unique values in each category
for col in categorical_cols:
    print(f"\nUnique values in '{col}':\n", df[col].value_counts())

#Encoding biinary categories
if 'churn' in df.columns:
    labenc = LabelEncoder()
    df['churn'] = labenc.fit_transform(df['churn'])
    print("\nEncoded 'churn' column successfully. Classes: ", labenc.classes_)


Categorical features found: ['country', 'gender']

Unique values in 'country':
 country
France     5014
Germany    2509
Spain      2477
Name: count, dtype: int64

Unique values in 'gender':
 gender
Male      5457
Female    4543
Name: count, dtype: int64

Encoded 'churn' column successfully. Classes:  [0 1]


### Normalizing and Scaling Continuous variables

In [22]:
#Identifying numberical features (other than churn)
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns
numerical_cols = [col for col in numerical_cols if col != 'churn']

#Scaling continuous features using StandardScaler
scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])

print(f"Sample rows of scaled features:\n {df[numerical_cols].head()}")

Sample rows of scaled features:
    customer_id  credit_score       age    tenure   balance  products_number  \
0    -0.783213     -0.326221  0.293517 -1.041760 -1.225848        -0.911583   
1    -0.606534     -0.440036  0.198164 -1.387538  0.117350        -0.911583   
2    -0.995885     -1.536794  0.293517  1.032908  1.333053         2.527057   
3     0.144767      0.501521  0.007457 -1.387538 -1.225848         0.807737   
4     0.652659      2.063884  0.388871 -1.041760  0.785728        -0.911583   

   credit_card  active_member  estimated_salary  
0     0.646092       0.970243          0.021886  
1    -1.547768       0.970243          0.216534  
2     0.646092      -1.030670          0.240687  
3    -1.547768      -1.030670         -0.108918  
4     0.646092       0.970243         -0.365276  


### Analyzing Class Imbalance (churners vs non-churners)

In [29]:
churn_counts = df['churn'].value_counts()
churn_percentages = df['churn'].value_counts(normalize=True) * 100

print("Class Imbalance Analysis:")
for idx, (cnt, pct) in enumerate(zip(churn_counts, churn_percentages)):
    print(f"Class {churn_counts.index[idx]}: {cnt:,} customers ({pct:.2f}%)")

#Flag to check implance
if churn_percentages.min() < 20:
    print("Heavy class imbalance")
else:
    print("Reasonable balance")

Class Imbalance Analysis:
Class 0: 7,963 customers (79.63%)
Class 1: 2,037 customers (20.37%)
Reasonable balance
